# Week 10 — Prueba con stakeholder, comparación baseline vs actual y latencia

## Proyecto

**Análisis de Sentimiento en Noticias Macroeconómicas mediante Inteligencia Artificial para la Generación de Señales de Apoyo a la Toma de Decisiones en Tesorería Bancaria**

## Objetivo del notebook

Este notebook genera la evidencia técnica solicitada para el sprint:

1. Comparación entre el **baseline estable** y el **modelo actual**.
2. Medición de latencia de inferencia: promedio, p50, p95 y p99.
3. Medición aproximada de throughput y ratio de errores.
4. Preparación y análisis de una prueba guiada con stakeholders.
5. Recomendación de despliegue mínimo basada en métricas técnicas y percepción de usuario.

### Definición experimental

- **Baseline:** Random Forest ganador con umbral de decisión 0.50.
- **Modelo actual:** mismo Random Forest con umbral mitigado obtenido en Week 9, aproximadamente 0.43.
- Se conserva el mismo conjunto de datos y las mismas probabilidades, de modo que la comparación mida exclusivamente el efecto del ajuste del umbral.

# 1. Importación de librerías y configuración

Se importan las librerías necesarias y se definen las rutas del proyecto. Los resultados se guardarán en `logs/`, `figs/` y `docs/data/`.

In [ ]:
from pathlib import Path
import json
import time
import platform
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix
)

warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

LOGS_DIR = PROJECT_ROOT / "logs"
FIGS_DIR = PROJECT_ROOT / "figs"
DOCS_DATA_DIR = PROJECT_ROOT / "docs" / "data"
ARTIFACTS_DIR = PROJECT_ROOT / "data" / "artifacts"

for folder in [LOGS_DIR, FIGS_DIR, DOCS_DATA_DIR, ARTIFACTS_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Python:", platform.python_version())
print("Sistema:", platform.platform())
print("Procesador:", platform.processor())

# 2. Carga de datos y configuración del modelo actual

Se utiliza el archivo exportado para el dashboard de Week 8/9. El notebook detecta automáticamente el umbral mitigado de Week 9 cuando existe un resumen JSON; de lo contrario utiliza 0.43.

In [ ]:
DATA_PATH = DOCS_DATA_DIR / "week8_dashboard_data.csv"
MODEL_PATH = ARTIFACTS_DIR / "week8_best_model.joblib"

if not DATA_PATH.exists():
    raise FileNotFoundError(f"No se encontró el dataset: {DATA_PATH}")

df = pd.read_csv(DATA_PATH)
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df = df.sort_values("date").reset_index(drop=True)

TARGET_COL = "target_up"
PROB_COL = "prob_up_rf"

required = [TARGET_COL, PROB_COL, "sent_index_mean"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Faltan columnas requeridas: {missing}")

df_eval = df.dropna(subset=[TARGET_COL, PROB_COL]).copy()
df_eval[TARGET_COL] = df_eval[TARGET_COL].astype(int)

# Buscar umbral mitigado de Week 9
threshold_candidates = [
    LOGS_DIR / "week9_summary.json",
    LOGS_DIR / "week9_error_analysis_summary.json",
    DOCS_DATA_DIR / "week9_summary.json"
]

CURRENT_THRESHOLD = 0.43

for p in threshold_candidates:
    if p.exists():
        with open(p, "r", encoding="utf-8") as f:
            obj = json.load(f)
        CURRENT_THRESHOLD = float(
            obj.get("mitigated_threshold",
            obj.get("best_threshold",
            obj.get("current_threshold", CURRENT_THRESHOLD)))
        )
        print("Umbral leído desde:", p)
        break

BASELINE_THRESHOLD = 0.50

print("Filas evaluables:", len(df_eval))
print("Umbral baseline:", BASELINE_THRESHOLD)
print("Umbral actual:", CURRENT_THRESHOLD)
display(df_eval.head())

# 3. Funciones de evaluación

Se definen funciones comunes para calcular Accuracy, Precision, Recall, F1, PR-AUC y los componentes de la matriz de confusión.

In [ ]:
def compute_metrics(y_true, y_score, threshold):
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    y_pred = (y_score >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "threshold": float(threshold),
        "n": int(len(y_true)),
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "pr_auc": average_precision_score(y_true, y_score)
            if len(np.unique(y_true)) > 1 else np.nan,
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
        "error_rate": float((fp + fn) / len(y_true))
    }

def bootstrap_metric_difference(
    y_true,
    y_score,
    threshold_a=0.50,
    threshold_b=0.43,
    metric="f1",
    n_boot=2000,
    seed=42
):
    rng = np.random.default_rng(seed)
    y_true = np.asarray(y_true).astype(int)
    y_score = np.asarray(y_score).astype(float)
    n = len(y_true)
    diffs = []

    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yt = y_true[idx]
        ys = y_score[idx]

        ma = compute_metrics(yt, ys, threshold_a)[metric]
        mb = compute_metrics(yt, ys, threshold_b)[metric]
        diffs.append(mb - ma)

    diffs = np.asarray(diffs, dtype=float)
    return {
        "metric": metric,
        "delta_mean": float(np.nanmean(diffs)),
        "delta_ci_low": float(np.nanquantile(diffs, 0.025)),
        "delta_ci_high": float(np.nanquantile(diffs, 0.975))
    }

# 4. Comparación técnica: baseline vs modelo actual

La comparación mantiene constantes el dataset, las probabilidades y el modelo. La única diferencia es el umbral de decisión.

- Baseline: umbral 0.50.
- Modelo actual: umbral mitigado de Week 9.

In [ ]:
y_true = df_eval[TARGET_COL].values
y_score = df_eval[PROB_COL].values

baseline_metrics = compute_metrics(y_true, y_score, BASELINE_THRESHOLD)
current_metrics = compute_metrics(y_true, y_score, CURRENT_THRESHOLD)

comparison_df = pd.DataFrame([
    {"variant": "Baseline RF threshold 0.50", **baseline_metrics},
    {"variant": f"Actual RF threshold {CURRENT_THRESHOLD:.2f}", **current_metrics}
])

metric_cols = [
    "accuracy", "precision", "recall", "f1", "pr_auc",
    "tn", "fp", "fn", "tp", "error_rate"
]

delta_row = {"variant": "Delta actual - baseline"}
for col in metric_cols:
    delta_row[col] = current_metrics[col] - baseline_metrics[col]

comparison_with_delta = pd.concat(
    [comparison_df, pd.DataFrame([delta_row])],
    ignore_index=True
)

display(comparison_with_delta)

# 5. Intervalo de confianza para la mejora

Se estima mediante bootstrap el intervalo de confianza del cambio en Recall y F1. Esto ayuda a distinguir una mejora observada de una fluctuación debida al tamaño reducido de la muestra.

In [ ]:
bootstrap_delta_rows = []

for metric in ["accuracy", "precision", "recall", "f1"]:
    bootstrap_delta_rows.append(
        bootstrap_metric_difference(
            y_true=y_true,
            y_score=y_score,
            threshold_a=BASELINE_THRESHOLD,
            threshold_b=CURRENT_THRESHOLD,
            metric=metric,
            n_boot=2000,
            seed=RANDOM_SEED
        )
    )

bootstrap_delta_df = pd.DataFrame(bootstrap_delta_rows)
display(bootstrap_delta_df)

# 6. Gráfico comparativo baseline vs actual

La figura permite comunicar rápidamente qué métricas mejoran y qué trade-off se produce al reducir el umbral.

In [ ]:
metrics_plot = ["accuracy", "precision", "recall", "f1", "pr_auc"]

x = np.arange(len(metrics_plot))
width = 0.36

fig, ax = plt.subplots(figsize=(9, 5))
base_vals = comparison_df.iloc[0][metrics_plot].astype(float).values
curr_vals = comparison_df.iloc[1][metrics_plot].astype(float).values

bars1 = ax.bar(x - width/2, base_vals, width, label="Baseline 0.50")
bars2 = ax.bar(x + width/2, curr_vals, width, label=f"Actual {CURRENT_THRESHOLD:.2f}")

ax.set_xticks(x)
ax.set_xticklabels(["Accuracy", "Precision", "Recall", "F1", "PR-AUC"])
ax.set_ylim(0, 1)
ax.set_ylabel("Valor")
ax.set_title("Comparación técnica: baseline vs modelo actual")
ax.legend()

for bars in [bars1, bars2]:
    for bar in bars:
        ax.text(
            bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.015,
            f"{bar.get_height():.3f}",
            ha="center",
            va="bottom",
            fontsize=9
        )

plt.tight_layout()
fig_path = FIGS_DIR / "week10_baseline_vs_current.png"
plt.savefig(fig_path, dpi=160, bbox_inches="tight")
plt.show()

print("Figura guardada en:", fig_path)

# 7. Preparación del modelo para perfilado de latencia

Se carga el artefacto Random Forest. Si las variables `lag_sent_1` y `lag_sent_2` no están en el CSV, se reconstruyen a partir del sentimiento diario.

La latencia medida corresponde a inferencia local del modelo, sin incluir red ni renderizado del dashboard.

In [ ]:
if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"No se encontró el modelo: {MODEL_PATH}. "
        "Ejecuta primero Week 8 o verifica la ruta."
    )

model = joblib.load(MODEL_PATH)

feature_cols = ["lag_sent_1", "lag_sent_2"]

if not set(feature_cols).issubset(df.columns):
    df_latency = df.copy()
    df_latency["lag_sent_1"] = df_latency["sent_index_mean"].shift(1)
    df_latency["lag_sent_2"] = df_latency["sent_index_mean"].shift(2)
else:
    df_latency = df.copy()

df_latency = df_latency.dropna(subset=feature_cols).copy()
X_latency = df_latency[feature_cols].astype(float)

print("Modelo cargado:", type(model).__name__)
print("Features:", feature_cols)
print("Filas para perfilado:", len(X_latency))
display(X_latency.head())

# 8. Warm-up y protocolo de medición

Antes de medir se ejecutan inferencias de calentamiento para reducir el efecto de inicialización. Luego se realizan múltiples repeticiones para obtener una distribución estable de latencias.

In [ ]:
# Warm-up
warmup_rows = min(10, len(X_latency))

for _ in range(30):
    sample = X_latency.sample(n=warmup_rows, replace=True, random_state=None)
    _ = model.predict_proba(sample)

print("Warm-up completado.")

# 9. Latencia por petición individual

Se mide la inferencia de una observación por vez. Las métricas principales son:

- p50: mediana de latencia.
- p95: el 95% de las peticiones respondió en ese tiempo o menos.
- p99: representa la cola más lenta.
- throughput: peticiones procesadas por segundo.
- ratio de errores: proporción de ejecuciones fallidas.

In [ ]:
def profile_single_request_latency(model, X, n_runs=2000, seed=42):
    rng = np.random.default_rng(seed)
    latencies_ms = []
    errors = 0

    for _ in range(n_runs):
        idx = int(rng.integers(0, len(X)))
        row = X.iloc[[idx]]

        start = time.perf_counter()
        try:
            _ = model.predict_proba(row)
        except Exception:
            errors += 1
        elapsed_ms = (time.perf_counter() - start) * 1000
        latencies_ms.append(elapsed_ms)

    latencies = np.asarray(latencies_ms, dtype=float)
    total_seconds = latencies.sum() / 1000

    return {
        "mode": "single_request",
        "batch_size": 1,
        "n_runs": int(n_runs),
        "mean_ms": float(latencies.mean()),
        "p50_ms": float(np.percentile(latencies, 50)),
        "p95_ms": float(np.percentile(latencies, 95)),
        "p99_ms": float(np.percentile(latencies, 99)),
        "min_ms": float(latencies.min()),
        "max_ms": float(latencies.max()),
        "throughput_req_s": float(n_runs / total_seconds) if total_seconds > 0 else np.nan,
        "errors": int(errors),
        "error_ratio": float(errors / n_runs),
        "latencies_ms": latencies
    }

single_profile = profile_single_request_latency(
    model=model,
    X=X_latency,
    n_runs=2000,
    seed=RANDOM_SEED
)

single_summary = {
    k: v for k, v in single_profile.items()
    if k != "latencies_ms"
}

display(pd.DataFrame([single_summary]))

# 10. Perfilado con micro-batching

Se evalúan tamaños de lote 1, 8 y 16. El objetivo es observar si agrupar solicitudes reduce el costo promedio por observación.

In [ ]:
def profile_batch_latency(model, X, batch_size, n_runs=500, seed=42):
    rng = np.random.default_rng(seed)
    batch_latencies_ms = []
    per_item_latencies_ms = []
    errors = 0
    total_items = 0

    for _ in range(n_runs):
        idx = rng.integers(0, len(X), size=batch_size)
        batch = X.iloc[idx]

        start = time.perf_counter()
        try:
            _ = model.predict_proba(batch)
        except Exception:
            errors += 1
        elapsed_ms = (time.perf_counter() - start) * 1000

        batch_latencies_ms.append(elapsed_ms)
        per_item_latencies_ms.append(elapsed_ms / batch_size)
        total_items += batch_size

    batch_latencies_ms = np.asarray(batch_latencies_ms, dtype=float)
    per_item_latencies_ms = np.asarray(per_item_latencies_ms, dtype=float)
    total_seconds = batch_latencies_ms.sum() / 1000

    return {
        "mode": "batch",
        "batch_size": int(batch_size),
        "n_runs": int(n_runs),
        "total_items": int(total_items),
        "mean_batch_ms": float(batch_latencies_ms.mean()),
        "p50_batch_ms": float(np.percentile(batch_latencies_ms, 50)),
        "p95_batch_ms": float(np.percentile(batch_latencies_ms, 95)),
        "p99_batch_ms": float(np.percentile(batch_latencies_ms, 99)),
        "mean_per_item_ms": float(per_item_latencies_ms.mean()),
        "p95_per_item_ms": float(np.percentile(per_item_latencies_ms, 95)),
        "throughput_req_s": float(total_items / total_seconds) if total_seconds > 0 else np.nan,
        "errors": int(errors),
        "error_ratio": float(errors / n_runs)
    }

batch_profiles = []

for batch_size in [1, 8, 16]:
    batch_profiles.append(
        profile_batch_latency(
            model=model,
            X=X_latency,
            batch_size=batch_size,
            n_runs=500,
            seed=RANDOM_SEED
        )
    )

batch_latency_df = pd.DataFrame(batch_profiles)
display(batch_latency_df)

# 11. Visualización de latencia y throughput

Se presentan p50, p95 y p99 para la inferencia individual, además del throughput obtenido con distintos tamaños de lote.

In [ ]:
latency_percentiles = pd.DataFrame({
    "percentile": ["p50", "p95", "p99"],
    "latency_ms": [
        single_profile["p50_ms"],
        single_profile["p95_ms"],
        single_profile["p99_ms"]
    ]
})

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(latency_percentiles["percentile"], latency_percentiles["latency_ms"])
ax.set_ylabel("Latencia (ms)")
ax.set_title("Latencia de inferencia individual")

for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f"{bar.get_height():.3f}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
latency_fig = FIGS_DIR / "week10_latency_percentiles.png"
plt.savefig(latency_fig, dpi=160, bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(7, 4.5))
bars = ax.bar(
    batch_latency_df["batch_size"].astype(str),
    batch_latency_df["throughput_req_s"]
)
ax.set_xlabel("Batch size")
ax.set_ylabel("Throughput (req/s)")
ax.set_title("Throughput según micro-batch")

for bar in bars:
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f"{bar.get_height():.1f}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
throughput_fig = FIGS_DIR / "week10_throughput_batch.png"
plt.savefig(throughput_fig, dpi=160, bbox_inches="tight")
plt.show()

print("Figuras guardadas:")
print("-", latency_fig)
print("-", throughput_fig)

# 12. Definición preliminar de SLO

Para esta etapa académica se propone un SLO de inferencia local:

- p95 menor a 50 ms por solicitud.
- ratio de errores menor a 0.5%.
- F1 del modelo actual no inferior al baseline.
- Recall del modelo actual superior al baseline.

Estos umbrales pueden revisarse después de conversar con el stakeholder.

In [ ]:
SLO_P95_MS = 50.0
SLO_ERROR_RATIO = 0.005

slo_results = pd.DataFrame([
    {
        "slo": "p95 inferencia < 50 ms",
        "observed": single_profile["p95_ms"],
        "target": SLO_P95_MS,
        "complies": single_profile["p95_ms"] < SLO_P95_MS
    },
    {
        "slo": "ratio errores < 0.5%",
        "observed": single_profile["error_ratio"],
        "target": SLO_ERROR_RATIO,
        "complies": single_profile["error_ratio"] < SLO_ERROR_RATIO
    },
    {
        "slo": "F1 actual >= baseline",
        "observed": current_metrics["f1"],
        "target": baseline_metrics["f1"],
        "complies": current_metrics["f1"] >= baseline_metrics["f1"]
    },
    {
        "slo": "Recall actual > baseline",
        "observed": current_metrics["recall"],
        "target": baseline_metrics["recall"],
        "complies": current_metrics["recall"] > baseline_metrics["recall"]
    }
])

display(slo_results)

# 13. Preparación de la prueba guiada con stakeholders

## Modo seleccionado

Se propone una prueba de **usabilidad guiada con comparación A/B controlada**:

- Variante A: baseline con umbral 0.50.
- Variante B: modelo actual con umbral mitigado.
- Participantes sugeridos: 3 a 5 personas con conocimiento de finanzas, tesorería, riesgo o analítica.
- Escenarios: caso típico, caso límite y caso de baja confianza.

El siguiente bloque crea una plantilla CSV. Debe completarse después de realizar las entrevistas. No deben inventarse respuestas.

In [ ]:
feedback_template_path = LOGS_DIR / "week10_stakeholder_feedback.csv"

feedback_columns = [
    "user_id",
    "role",
    "variant",
    "scenario_id",
    "utility_1_5",
    "clarity_1_5",
    "confidence_1_5",
    "task_success",
    "task_time_seconds",
    "preferred_variant",
    "comments",
    "never_use_reason"
]

if not feedback_template_path.exists():
    template = pd.DataFrame(columns=feedback_columns)
    template.to_csv(
        feedback_template_path,
        index=False,
        encoding="utf-8-sig"
    )
    print("Plantilla creada en:", feedback_template_path)
else:
    print("La plantilla ya existe:", feedback_template_path)

display(pd.read_csv(feedback_template_path).head())

# 14. Guion de prueba con stakeholder

Para cada participante se recomienda seguir este flujo:

1. Explicar brevemente el objetivo del sistema.
2. Mostrar Variante A y Variante B sin indicar inicialmente cuál es la nueva.
3. Pedir que interprete tres escenarios:
   - señal clara de subida;
   - señal cercana al umbral;
   - señal donde el modelo se equivocó.
4. Medir tiempo de tarea y éxito.
5. Solicitar calificaciones de utilidad, claridad y confianza.
6. Registrar comentarios y situaciones en las que no usaría la recomendación.

Las respuestas deben guardarse en `logs/week10_stakeholder_feedback.csv`.

# 15. Análisis de percepción del usuario

Este bloque solo genera resultados cuando el CSV contiene respuestas válidas. Si aún está vacío, el notebook informa que la prueba está pendiente.

In [ ]:
feedback_df = pd.read_csv(feedback_template_path)

rating_cols = ["utility_1_5", "clarity_1_5", "confidence_1_5"]

if feedback_df.empty:
    print(
        "La prueba con stakeholders todavía está pendiente. "
        "Completa el CSV y vuelve a ejecutar esta celda."
    )
    stakeholder_summary = pd.DataFrame()
else:
    for col in rating_cols + ["task_time_seconds"]:
        feedback_df[col] = pd.to_numeric(feedback_df[col], errors="coerce")

    feedback_df["task_success"] = (
        feedback_df["task_success"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": 1, "1": 1, "sí": 1, "si": 1, "yes": 1,
            "false": 0, "0": 0, "no": 0
        })
    )

    stakeholder_summary = (
        feedback_df.groupby("variant")
        .agg(
            n_responses=("user_id", "count"),
            utility_mean=("utility_1_5", "mean"),
            clarity_mean=("clarity_1_5", "mean"),
            confidence_mean=("confidence_1_5", "mean"),
            task_success_rate=("task_success", "mean"),
            task_time_mean_seconds=("task_time_seconds", "mean")
        )
        .reset_index()
    )

    display(stakeholder_summary)

# 16. Visualización de percepción del usuario

La figura compara utilidad, claridad y confianza entre ambas variantes. Se genera únicamente cuando existen respuestas registradas.

In [ ]:
stakeholder_fig = FIGS_DIR / "week10_user_perception.png"

if stakeholder_summary.empty:
    print("No se genera figura porque todavía no existen respuestas.")
else:
    perception_cols = ["utility_mean", "clarity_mean", "confidence_mean"]
    labels = ["Utilidad", "Claridad", "Confianza"]

    x = np.arange(len(labels))
    variants = stakeholder_summary["variant"].tolist()
    width = 0.8 / max(len(variants), 1)

    fig, ax = plt.subplots(figsize=(8, 5))

    for i, (_, row) in enumerate(stakeholder_summary.iterrows()):
        offset = (i - (len(variants) - 1) / 2) * width
        vals = row[perception_cols].astype(float).values
        bars = ax.bar(x + offset, vals, width, label=str(row["variant"]))

        for bar in bars:
            ax.text(
                bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.05,
                f"{bar.get_height():.2f}",
                ha="center",
                va="bottom",
                fontsize=9
            )

    ax.set_xticks(x)
    ax.set_xticklabels(labels)
    ax.set_ylim(0, 5)
    ax.set_ylabel("Promedio (1–5)")
    ax.set_title("Percepción del usuario: baseline vs actual")
    ax.legend()

    plt.tight_layout()
    plt.savefig(stakeholder_fig, dpi=160, bbox_inches="tight")
    plt.show()

    print("Figura guardada en:", stakeholder_fig)

# 17. Recomendación de despliegue mínimo

La decisión combina evidencia técnica, desempeño operativo y percepción del usuario.

La recomendación se clasifica en:

- `GO_CONTROLLED_DEMO`: cumple requisitos técnicos y puede pasar a una demo controlada.
- `GO_WITH_CONDITIONS`: técnicamente viable, pero faltan respuestas o existen observaciones.
- `NO_GO`: no cumple requisitos mínimos.

In [ ]:
technical_ok = bool(
    single_profile["p95_ms"] < SLO_P95_MS
    and single_profile["error_ratio"] < SLO_ERROR_RATIO
    and current_metrics["f1"] >= baseline_metrics["f1"]
    and current_metrics["recall"] > baseline_metrics["recall"]
)

if stakeholder_summary.empty:
    decision = "GO_WITH_CONDITIONS" if technical_ok else "NO_GO"
    decision_reason = (
        "El modelo cumple los criterios técnicos preliminares, "
        "pero falta completar la prueba con stakeholders."
        if technical_ok else
        "El modelo no cumple uno o más criterios técnicos y además "
        "falta completar la prueba con stakeholders."
    )
else:
    current_rows = stakeholder_summary[
        stakeholder_summary["variant"].astype(str).str.lower().str.contains("actual|b")
    ]

    if current_rows.empty:
        user_ok = False
        user_note = "No se identificó una fila correspondiente a la variante actual."
    else:
        current_user = current_rows.iloc[0]
        user_ok = bool(
            current_user["utility_mean"] >= 4.0
            and current_user["clarity_mean"] >= 4.0
            and current_user["confidence_mean"] >= 3.5
            and current_user["task_success_rate"] >= 0.80
        )
        user_note = "Se evaluaron utilidad, claridad, confianza y éxito de tarea."

    if technical_ok and user_ok:
        decision = "GO_CONTROLLED_DEMO"
        decision_reason = "Cumple criterios técnicos y de percepción de usuario."
    elif technical_ok:
        decision = "GO_WITH_CONDITIONS"
        decision_reason = (
            "Cumple criterios técnicos, pero la percepción de usuario "
            "requiere ajustes antes de una demo más amplia."
        )
    else:
        decision = "NO_GO"
        decision_reason = "No cumple los criterios técnicos mínimos."

decision_df = pd.DataFrame([{
    "decision": decision,
    "technical_ok": technical_ok,
    "reason": decision_reason
}])

display(decision_df)

# 18. Exportación de evidencias

Se exportan las tablas necesarias para el informe y GitHub:

- comparativo baseline vs actual;
- intervalos de confianza de la mejora;
- resultados de latencia;
- resultados de micro-batching;
- cumplimiento de SLO;
- resumen de percepción del usuario;
- recomendación final.

In [ ]:
comparison_path = LOGS_DIR / "week10_baseline_vs_current.csv"
bootstrap_path = LOGS_DIR / "week10_baseline_vs_current_bootstrap_ci.csv"
latency_path = LOGS_DIR / "week10_latency_results.csv"
batch_path = LOGS_DIR / "week10_batch_latency_results.csv"
slo_path = LOGS_DIR / "week10_slo_results.csv"
stakeholder_path = LOGS_DIR / "week10_stakeholder_summary.csv"
decision_path = LOGS_DIR / "week10_deployment_recommendation.csv"

comparison_with_delta.to_csv(comparison_path, index=False, encoding="utf-8-sig")
bootstrap_delta_df.to_csv(bootstrap_path, index=False, encoding="utf-8-sig")

single_export = pd.DataFrame([single_summary])
single_export.to_csv(latency_path, index=False, encoding="utf-8-sig")

batch_latency_df.to_csv(batch_path, index=False, encoding="utf-8-sig")
slo_results.to_csv(slo_path, index=False, encoding="utf-8-sig")
stakeholder_summary.to_csv(stakeholder_path, index=False, encoding="utf-8-sig")
decision_df.to_csv(decision_path, index=False, encoding="utf-8-sig")

# Copias para GitHub Pages / dashboard
comparison_with_delta.to_csv(
    DOCS_DATA_DIR / "week10_baseline_vs_current.csv",
    index=False,
    encoding="utf-8-sig"
)
single_export.to_csv(
    DOCS_DATA_DIR / "week10_latency_results.csv",
    index=False,
    encoding="utf-8-sig"
)
batch_latency_df.to_csv(
    DOCS_DATA_DIR / "week10_batch_latency_results.csv",
    index=False,
    encoding="utf-8-sig"
)
stakeholder_summary.to_csv(
    DOCS_DATA_DIR / "week10_stakeholder_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Archivos exportados:")
for p in [
    comparison_path,
    bootstrap_path,
    latency_path,
    batch_path,
    slo_path,
    stakeholder_path,
    decision_path
]:
    print("-", p)

# 19. Resumen automático para el informe

Este texto resume los resultados técnicos disponibles. La parte de stakeholder se completa después de registrar respuestas reales.

In [ ]:
print(
    f'''
COMPARACIÓN TÉCNICA
-------------------
Baseline:
- Threshold: {BASELINE_THRESHOLD:.2f}
- Accuracy: {baseline_metrics["accuracy"]:.3f}
- Precision: {baseline_metrics["precision"]:.3f}
- Recall: {baseline_metrics["recall"]:.3f}
- F1: {baseline_metrics["f1"]:.3f}
- PR-AUC: {baseline_metrics["pr_auc"]:.3f}
- FN: {baseline_metrics["fn"]}

Modelo actual:
- Threshold: {CURRENT_THRESHOLD:.2f}
- Accuracy: {current_metrics["accuracy"]:.3f}
- Precision: {current_metrics["precision"]:.3f}
- Recall: {current_metrics["recall"]:.3f}
- F1: {current_metrics["f1"]:.3f}
- PR-AUC: {current_metrics["pr_auc"]:.3f}
- FN: {current_metrics["fn"]}

LATENCIA LOCAL
--------------
- p50: {single_profile["p50_ms"]:.3f} ms
- p95: {single_profile["p95_ms"]:.3f} ms
- p99: {single_profile["p99_ms"]:.3f} ms
- Throughput: {single_profile["throughput_req_s"]:.1f} req/s
- Ratio de errores: {single_profile["error_ratio"]:.4%}

RECOMENDACIÓN
-------------
- Decisión preliminar: {decision}
- Motivo: {decision_reason}
'''
)

# 20. Conclusiones del sprint

La comparación debe interpretarse considerando tres dimensiones:

1. **Desempeño técnico:** determinar si el modelo actual mejora Recall y F1 respecto al baseline.
2. **Desempeño operativo:** verificar si la latencia p95 y el ratio de errores cumplen el SLO.
3. **Utilidad para el usuario:** validar si stakeholders perciben la salida como útil, clara y confiable.

La versión actual solo debe recomendarse para una demo controlada o modo sombra. No debe presentarse como sistema de decisión automática ni como herramienta lista para producción bancaria.